# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** *Twilight Imperium 4th Edition: Prophecy of Kings, Living Rules Reference 2.0* (44 pages, a dense board-game rulebook).

Two pipelines run over the **same chunked corpus**:

| | Retrieval | Passed to the LLM |
|---|---|---|
| **Pipeline 1 (baseline)** | Bi-encoder → FAISS → top 10 by cosine similarity | top 3 straight from the bi-encoder |
| **Pipeline 2 (re-ranking)** | the *same* top 10 → Cross-Encoder scores all 10 `(query, chunk)` pairs | top 3 after re-sorting by cross-encoder score |

Both pipelines use an identical prompt, and the LLM runs at temperature 0 with a fixed seed, so the difference between Answer 1 and Answer 2 comes from which chunks were retrieved. (The wording of an answer can still vary slightly from run to run; see Section 6.)

Notebook outline: setup → PDF loading → chunking → indexing → retrieval → prompt/LLM → `run_both` → failure-query discovery → rank-shift tables → answer comparison → latency profiling → export.

## 1. Setup
Models, constants and seeds. The bi-encoder and cross-encoder are small MiniLM models. The bi-encoder is deliberately a weak-ish one, which makes retrieval failures easier to observe. The LLM is a local open-weights model served by Ollama.

In [1]:
import os, re, time, json, random, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch, faiss, pymupdf, ollama
from sentence_transformers import SentenceTransformer, CrossEncoder

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PDF_URL  = ("https://images-cdn.fantasyflightgames.com/filer_public/51/55/"
            "51552c7f-c05c-445b-84bf-4b073456d008/ti10_pok_living_rules_reference_20_web.pdf")
PDF_PATH = Path("data/ti4_lrr.pdf")

BI_ENCODER    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM           = "qwen2.5:7b"          # local, served by Ollama

TOP_K, FINAL_K = 10, 3                # retrieve 10 candidates, give 3 to the LLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for d in ("data", "results", "report/tables"):
    Path(d).mkdir(parents=True, exist_ok=True)
print("device:", DEVICE)

device: cuda


## 2. Load the PDF
The rulebook uses a **two-column layout**. Plain text extraction interleaves the columns, so instead I take PyMuPDF's text *blocks* and sort them so that the left column is read top-to-bottom, then the right column. Page-number blocks are dropped.

In [2]:
if not PDF_PATH.exists():             # the PDF is copyrighted so it is not stored in the repo
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    PDF_PATH.write_bytes(urllib.request.urlopen(req, timeout=60).read())

def read_blocks(pdf_path):
    """Return [(page_number, raw_block_text)] in reading order (left column, then right column)."""
    out = []
    for pno, page in enumerate(pymupdf.open(pdf_path), start=1):
        mid = page.rect.width / 2
        blocks = [b for b in page.get_text("blocks") if b[6] == 0]       # b[6]==0 -> text block
        blocks.sort(key=lambda b: (b[0] >= mid - 20, round(b[1]), b[0])) # column, then y, then x
        for b in blocks:
            if re.fullmatch(r"\s*\d+\s*", b[4]):                         # bare page number
                continue
            out.append((pno, b[4]))
    return out

blocks = read_blocks(PDF_PATH)
print(f"{len(blocks)} text blocks from {blocks[-1][0]} pages")

2104 text blocks from 44 pages


## 3. Chunking
The rulebook is organised as numbered **topics** (`37 FLEET POOL`) containing numbered **sub-rules** (`37.3`) with lettered sub-clauses (a, b, …). I chunk along that structure so each chunk is one rule, which makes retrieved chunks readable and lets me name them, e.g. `Fleet Pool 37.3`.

Page ranges of the document:
* pages 1–3: cover, changelog and a list of changed rules (skipped)
* pages 4–5: introduction and game setup → chunks `Intro`, `Setup-Step N`
* page 6: board-layout diagrams (skipped, only labels)
* pages 7–37: the glossary of numbered rules → chunks `NN.N`
* pages 38–39: alphabetical **index** (skipped: it is a bare list of terms, which would act as a fake keyword magnet)
* pages 40–44: errata and FAQ → chunks `Errata-X` and `FAQ-N`

Choices:
* Each chunk's text starts with its topic title, so a lone rule like "A player cannot…" keeps its subject.
* The `RELATED TOPICS:` navigation lines are dropped. They are cross-references, not rules.
* Chunks shorter than 25 words are merged into the next rule of the same topic. Chunks longer than 300 words are split at sentence boundaries.

In [3]:
HEAD = re.compile(r"^(\d{1,3}) ([A-Z][A-Z0-9 ()'’/&-]+)$")   # "37 FLEET POOL"
RULE = re.compile(r"^(\d{1,3}\.\d{1,2}) (.*)$", re.S)        # "37.3 ..."
CAPS = re.compile(r"^[A-Z][A-Z0-9 ()'’/&:,.\"-]+$")          # all-caps label such as "GENERAL"
STEP = re.compile(r"^STEP \d+")

def norm(s):
    return re.sub(r"\s+", " ", s).strip()

def nice(title):
    return title.title().replace("Pds", "PDS")

def build_chunks(blocks):
    chunks = []                                   # dicts: id, topic, page, text
    cur = None

    def flush():
        nonlocal cur
        if cur and len(cur["text"].split()) > 1:
            chunks.append(cur)
        cur = None

    def start(cid, topic, page, text=""):
        nonlocal cur
        flush()
        cur = {"id": cid, "topic": topic, "page": page, "text": text}

    topic, faq_n, mode, label = None, 0, "front", ""
    for page, raw in blocks:
        t = norm(raw)
        if not t:
            continue
        # ---- which part of the document are we in? ----
        if page <= 3 or page == 6 or page in (38, 39):
            continue
        if page in (4, 5):
            mode = "setup"
        elif 7 <= page <= 37:
            mode = "glossary"
        elif page >= 40 and mode not in ("errata", "faq"):
            mode = "errata"

        if mode == "setup":
            if cur is None:
                start("Intro", "Introduction", page)
            if STEP.match(t):
                start(f"Setup-Step{re.match(r'STEP (\d+)', t).group(1)}", "Game Setup", page)
            cur["text"] += " " + t
        elif mode == "glossary":
            m_head, m_rule = HEAD.match(t), RULE.match(t)
            if m_head:
                topic = nice(m_head.group(2))
                start(f"{m_head.group(1)}.0", topic, page)
            elif t.startswith("RELATED TOPICS"):
                continue                          # navigation only
            elif m_rule:
                start(m_rule.group(1), topic, page, m_rule.group(2))
            elif cur is not None:
                # lettered sub-clause ("a<em-space>...") or continuation of the previous block
                m_sub = re.match(r"^([a-z]) ", raw.lstrip())
                cur["text"] += (f" ({m_sub.group(1)}) " + t[2:].strip()) if m_sub else " " + t
        else:  # errata / FAQ
            # an all-caps heading often shares a block with the text after it ("THE MENTAK COALITION\nQ: ...")
            first, _, rest = raw.strip().partition("\n")
            pieces = [first, rest] if CAPS.match(norm(first)) and rest.strip() else [raw]
            for piece in pieces:
                t = norm(piece)
                if t == "FAQ":
                    mode, label = "faq", "General"
                    flush(); continue
                if t == "ERRATA":
                    mode = "errata"; continue
                if CAPS.match(t) and len(t) > 2:      # all-caps label: errata item or FAQ section
                    label = nice(t)
                    if mode == "errata":
                        start(f"Errata-{label}", label, page)
                    else:
                        flush()
                    continue
                if mode == "faq" and t.startswith("Q:"):
                    faq_n += 1
                    start(f"FAQ-{faq_n}", label, page, t)
                elif cur is not None:
                    cur["text"] += " " + t
    flush()
    return chunks

raw_chunks = build_chunks(blocks)
print(len(raw_chunks), "raw chunks")

698 raw chunks


### Merge tiny chunks, split huge ones, add the topic title

In [4]:
MIN_W, MAX_W = 25, 300

def wc(t): return len(t.split())

def merge_and_split(chunks):
    # 1) merge chunks under MIN_W words into the next chunk of the same topic
    merged, i = [], 0
    while i < len(chunks):
        c = dict(chunks[i])
        while (wc(c["text"]) < MIN_W and i + 1 < len(chunks)
               and chunks[i + 1]["topic"] == c["topic"]):
            nxt = chunks[i + 1]
            c["id"]    = f'{c["id"]}–{nxt["id"]}'
            c["text"] += " " + nxt["text"]
            i += 1
        merged.append(c); i += 1
    # 2) split chunks over MAX_W words at sentence boundaries
    final = []
    for c in merged:
        if wc(c["text"]) <= MAX_W:
            final.append(c); continue
        part, buf = 1, []
        for s in re.split(r"(?<=[.!?]) ", c["text"]):
            if buf and wc(" ".join(buf + [s])) > MAX_W:
                final.append({**c, "id": f'{c["id"]}#{part}', "text": " ".join(buf)}); part += 1; buf = []
            buf.append(s)
        final.append({**c, "id": f'{c["id"]}#{part}', "text": " ".join(buf)})
    return final

chunks = merge_and_split(raw_chunks)
for c in chunks:
    c["name"] = f'{c["id"]} {c["topic"]}'        # readable label used in all tables, e.g. "37.3 Fleet Pool"
    c["text"] = f'{c["topic"]}: {norm(c["text"])}'  # topic title prefix keeps a rule attached to its subject
by_name = {c["name"]: i for i, c in enumerate(chunks)}
assert len(by_name) == len(chunks), "chunk names must be unique"

lens = pd.Series([wc(c["text"]) for c in chunks])
print(f"{len(chunks)} chunks | words per chunk: mean {lens.mean():.0f}, median {lens.median():.0f}, min {lens.min()}, max {lens.max()}")
for c in chunks[40:43] + chunks[-2:]:
    print(f'\n[{c["name"]}] (p.{c["page"]})\n{c["text"][:300]}')

511 chunks | words per chunk: mean 54, median 45, min 9, max 301

[3.2–3.3 Action Phase] (p.8)
Action Phase: If a player cannot perform an action, they must pass. After a player has passed, they have no further turns and cannot perform additional actions during that action phase. (a) During a turn that a player passes, they can resolve transactions and “at the start of your turn” abilities. (

[3.4 Action Phase] (p.8)
Action Phase: A player cannot pass until they have performed the strategic action of their strategy card. (a) During a three-player or four-player game, a player cannot pass until they have exhausted both of their strategy cards.

[3.5 Action Phase] (p.8)
Action Phase: After all players have passed, play proceeds to the status phase.

[FAQ-68 The Mentak Coalition] (p.44)
The Mentak Coalition: Q: Can the Mentak Coalition use the “Pillage” faction ability to steal trade goods when they are neighbors with another faction as a result of the “Lazax Gate Folding” faction techno

## 4. Indexing (Pipeline 1a)
Every chunk is embedded by the bi-encoder **independently** of any query. Vectors are L2-normalised, so the inner product equals cosine similarity, and stored in a FAISS flat inner-product index (exact search).

In [5]:
bi = SentenceTransformer(BI_ENCODER, device=DEVICE)
ce = CrossEncoder(CROSS_ENCODER, device=DEVICE)

texts = [c["text"] for c in chunks]
emb = bi.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True,
                show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("indexed", index.ntotal, "chunks, dim", emb.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 30208.61it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15246.73it/s]


BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


indexed 511 chunks, dim 384


## 5. Retrieval
**Pipeline 1 (baseline):** embed the query with the same bi-encoder, search FAISS for the top 10 by cosine similarity. The first 3 are what Pipeline 1 passes to the LLM.

**Pipeline 2 (re-ranking):** take those *same* 10 candidates and score every `(query, chunk)` pair with the cross-encoder. Unlike the bi-encoder, the cross-encoder reads query and chunk **together** in one transformer pass, so its attention can compare words across the two texts (negations, "unless", exact terms). Sort by score, descending, keep the top 3.

In [6]:
def embed_query(query):
    return bi.encode([query], normalize_embeddings=True, convert_to_numpy=True,
                     show_progress_bar=False).astype("float32")

def search(qvec, k=TOP_K):
    """Pipeline 1b: FAISS inner product over normalised vectors == cosine similarity."""
    scores, ids = index.search(qvec, k)
    return [{"idx": int(i), "bi_rank": r, "bi_score": float(s)}
            for r, (i, s) in enumerate(zip(ids[0], scores[0]), start=1)]

def rerank(query, candidates):
    """Pipeline 2b/2c: cross-encode all (query, chunk) pairs, re-sort descending by score."""
    pairs  = [(query, chunks[c["idx"]]["text"]) for c in candidates]
    scores = ce.predict(pairs, batch_size=len(pairs), show_progress_bar=False)
    out = sorted(({**c, "ce_score": float(s)} for c, s in zip(candidates, scores)),
                 key=lambda c: -c["ce_score"])
    for r, c in enumerate(out, start=1):
        c["ce_rank"] = r
    return out

## 6. One shared prompt and the LLM call
Both pipelines use the **same prompt template**; only the chunks differ. Temperature 0 and a fixed seed make generation close to deterministic: in a repeat run from a fresh clone of the repo, the wording of some answers changed slightly (GPU arithmetic is not bit-for-bit reproducible), but every answer reached the same conclusion. The model is told to cite chunk ids, so it is visible which chunk an answer relied on, and to say so if the excerpts do not contain the answer.

In [7]:
PROMPT = """You are a rules assistant for the board game Twilight Imperium 4th Edition.
Answer the question using ONLY the rule excerpts below, in one to three sentences, and cite the ids of the excerpts you used in square brackets, e.g. [37.3 Fleet Pool].
If the excerpts do not contain the rule needed to answer, say that the answer is not in the excerpts. Do not use outside knowledge.

Excerpts:
{context}

Question: {question}
Answer:"""

def generate(question, idxs):
    context = "\n\n".join(f'[{chunks[i]["name"]}] {chunks[i]["text"]}' for i in idxs)
    resp = ollama.chat(model=LLM,
                       messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
                       options={"temperature": 0, "seed": SEED, "num_predict": 300})
    return resp["message"]["content"].strip()

## 7. `run_both`: both pipelines on one query
Runs the full flow and times every stage. `timed()` synchronises the GPU before and after the call so that asynchronous CUDA work is included in the measurement.

In [8]:
def timed(fn, *args):
    if DEVICE == "cuda": torch.cuda.synchronize()
    t = time.perf_counter()
    result = fn(*args)
    if DEVICE == "cuda": torch.cuda.synchronize()
    return result, time.perf_counter() - t

def run_both(query, with_llm=True):
    qvec, t_embed  = timed(embed_query, query)
    cands, t_search = timed(search, qvec)                  # top-10, shared by both pipelines
    reranked, t_ce  = timed(rerank, query, cands)

    top3_base = [c["idx"] for c in cands[:FINAL_K]]        # Pipeline 1c
    top3_rr   = [c["idx"] for c in reranked[:FINAL_K]]     # Pipeline 2c
    res = {"query": query, "cands": cands, "reranked": reranked,
           "top3_base": top3_base, "top3_rr": top3_rr,
           "t": {"embed": t_embed, "search": t_search, "cross_encoder": t_ce}}
    if with_llm:
        res["answer1"], res["t"]["llm1"] = timed(generate, query, top3_base)
        res["answer2"], res["t"]["llm2"] = timed(generate, query, top3_rr)
    return res

def rank_table(res):
    """Rank-shift table for the 10 candidates: bi-encoder rank/score vs cross-encoder rank/score."""
    df = pd.DataFrame(res["reranked"])
    df["chunk"] = [chunks[i]["name"] for i in df["idx"]]
    df["shift"] = df["bi_rank"] - df["ce_rank"]            # positive = promoted by the cross-encoder
    return (df.sort_values("bi_rank")[["chunk", "bi_rank", "bi_score", "ce_rank", "ce_score", "shift"]]
              .reset_index(drop=True))

## 8. Failure-query discovery
I do not know in advance which questions will break the baseline, so I searched for them. I wrote **153 candidate questions** about the rulebook, in plain player language (players rarely use the glossary's exact wording), spread over the failure types the assignment lists:

* **keyword** – keyword trap / distractor: many chunks share the query's words, the answer uses other terms
* **exception** – a general rule is found but the restriction / "except when" sits in another chunk
* **negation** – "can't / not allowed / prohibited" questions
* **role** – who may do what (speaker, active player, defender …)
* **buried** – a specific detail hidden inside a long chunk

For each question I **hand-labelled the gold chunk(s)**, meaning the chunk(s) that contain the rule needed for a correct answer, by reading the rulebook text (not by looking at model output). The notebook then records where the best gold chunk lands in the baseline ranking and after re-ranking.

A question counts as a **failure case fixed by re-ranking** when the gold chunk is at baseline rank **4–10** (so it never reaches the LLM) but the cross-encoder lifts it into the **top 3**.

In [9]:
# (category, question, gold chunk ids). A gold id matches a chunk if it equals any part of the chunk id,
# because tiny neighbouring rules were merged into ranges such as "37.0–37.1".
CANDIDATES = [
    ('keyword', 'How do I take over Mecatol Rex?', ['27.2', '54.0']),
    ('keyword', 'When can I get back a ship that an opponent took from me?', ['17.2']),
    ('keyword', 'How many ships am I allowed to keep in one system?', ['37.0', '76.2']),
    ('keyword', 'How many victory points do I need to win the game?', ['98.0', '98.2']),
    ('keyword', 'Which planets can be explored for new cards?', ['35.2']),
    ('keyword', 'What happens to my fighters when there is no room for them on my carriers?', ['16.3', '36.2']),
    ('keyword', 'How do I cast votes?', ['8.5']),
    ('keyword', 'What does it cost to put down a defensive structure with the construction card?', ['24.2', '26.2']),
    ('exception', 'Does every ship count toward my fleet pool limit?', ['76.2', '37.0']),
    ('exception', 'Can I build ships in a system where an opponent has ships?', ['67.6', '14.1']),
    ('exception', 'Do ground forces count against my ship capacity?', ['16.3']),
    ('exception', 'Can I use sustain damage to cancel a hit from anti-fighter barrage?', ['87.3']),
    ('exception', 'Can I retreat into any adjacent system I want?', ['78.7']),
    ('exception', 'Can a ship move if it starts in a system that has my command token?', ['58.4']),
    ('exception', 'Do I need to be neighbors to make a deal with another player?', ['28.0', '94.5']),
    ('exception', 'Can I score more than one public objective per status phase?', ['61.6']),
    ('exception', 'Can a player who has already passed still do anything?', ['3.2']),
    ('exception', 'What happens if both players lose all their ground forces in a fight over a planet?', ['49.5', '42.4']),
    ('exception', 'Can a war sun bombard a planet that has a planetary shield?', ['65.2']),
    ('exception', 'Do I always roll a die for ships leaving a gravity rift?', ['41.2']),
    ('negation', 'Can my ships fly through an asteroid field?', ['11.0']),
    ('negation', 'Can a ship move through a nebula?', ['59.0']),
    ('negation', 'Can I use the primary ability of a strategy card another player chose?', ['83.3']),
    ('negation', 'Can I spend trade goods to cast votes?', ['8.12', '47.3', '93.3']),
    ('negation', "Can I score another player's secret objective?", ['61.19']),
    ('negation', "Am I allowed to play my own faction's promissory note?", ['69.2']),
    ('negation', 'Can relics be traded to another player?', ['73.3']),
    ('negation', 'Can I activate a system that already has my command token in it?', ['5.2', '89.1']),
    ('negation', 'Which cards am I not allowed to exchange in a transaction?', ['94.3']),
    ('negation', 'Can I produce units on a planet I do not control?', ['67.4']),
    ('negation', 'Can a commander be exhausted?', ['51.7']),
    ('negation', 'Are ground forces allowed to land on Mecatol Rex at the start of the game?', ['27.1']),
    ('negation', 'Can I place a third PDS on one planet?', ['63.2', '85.5']),
    ('negation', 'Can I make a unit that has no cost?', ['26.2']),
    ('role', 'Who decides what happens when the votes are tied?', ['8.18', '80.2']),
    ('role', 'Can the defender announce a retreat if the attacker already did?', ['78.4']),
    ('role', "Does the speaker's tie-break count as a vote?", ['8.18']),
    ('role', 'What happens to the speaker token when the speaker is eliminated?', ['33.8', '80.7']),
    ('buried', 'What happens to a ship that leaves a gravity rift?', ['41.0', '41.2']),
    ('buried', 'How much influence do I pay for each extra command token with Leadership?', ['52.2']),
    ('buried', 'How many resources does it take to research a second technology with the Technology card?', ['91.2']),
    ('buried', 'How many trade goods does the Trade card give?', ['92.2']),
    ('buried', 'What is the maximum number of secret objectives I can hold?', ['61.21', '45.4']),
    ('buried', 'How does a nebula help the defender in combat?', ['59.3']),
    ('buried', 'How many command tokens do I get back at the end of the status phase?', ['81.5']),
    ('buried', 'What does a zero on the ten-sided die mean?', ['FAQ-1']),
    ('buried', 'What happens when a space combat is impossible to win for both sides?', ['FAQ-2']),
    ('buried', 'Can fighters stop my ships from moving?', ['FAQ-3']),
    ('buried', 'Is the Planetary Shield stopping the X-89 Bacterial Weapon?', ['FAQ-5', '65.0']),
    ('buried', 'Do I pay for units produced by effects other than Production?', ['FAQ-17']),
    ('keyword', 'What happens if two players are tied for the most victory points when the game ends?', ['98.6', '98.8', '61.15']),
    ('keyword', 'Can I spend resources from a planet that is exhausted?', ['64.8']),
    ('keyword', 'Is a planet adjacent to the system it sits in?', ['6.3']),
    ('keyword', 'What do I do when I have more fighters than my carriers can hold after the battle?', ['16.3', '78.10']),
    ('keyword', 'Do units sitting on planets count toward my fleet pool?', ['37.0']),
    ('keyword', 'What is the trait of a planet good for?', ['64.2']),
    ('keyword', 'What can a player who is not part of a combat do against the participants?', ['62.1']),
    ('keyword', 'How many dice does a PDS roll when it shoots?', ['77.3', '77.7']),
    ('keyword', 'What happens to a captured infantry unit?', ['17.3', '17.4']),
    ('keyword', 'Who is allowed to see my hand of action cards?', ['2.5']),
    ('keyword', 'How many votes does one planet give me?', ['8.5', '47.1']),
    ('keyword', 'If my enemy parks ships around my shipyard, can I still build there?', ['14.1', '67.6']),
    ('keyword', 'When does my prisoner of war get released?', ['17.2', '17.4']),
    ('keyword', 'Can I make a trade with someone on the other side of the galaxy?', ['60.1', '94.0', '28.0', '94.5']),
    ('keyword', 'Which phase do I draw action cards in?', ['2.1', '81.3']),
    ('keyword', 'What do I get for destroying all enemy structures on a planet by invading it?', ['49.5']),
    ('keyword', 'Does a PDS get destroyed when an enemy lands on its planet and I have no troops there?', ['63.4']),
    ('keyword', 'If I lose my last troops on a planet and an enemy has some, who owns it?', ['25.5']),
    ('exception', 'Can I move a ship out of a system when an enemy ship sits in the way?', ['58.4']),
    ('exception', 'Can I put a space dock on every planet I own?', ['79.2', '85.2']),
    ('exception', 'Does a damaged unit fight worse than a healthy one?', ['87.2']),
    ('exception', 'If the planet I invade has no defenders, do I still need to win a fight to take it?', ['49.5', '49.4']),
    ('exception', 'What happens to structures on a planet when someone else takes control of it?', ['49.5']),
    ('exception', 'Does having a command token in a system stop me from picking up units there?', ['95.2']),
    ('exception', 'Can I use Space Cannon when I am not the active player?', ['77.1']),
    ('exception', 'What if I run out of plastic infantry pieces when I produce more?', ['67.5', '23.3']),
    ('exception', 'Do I still get to use Production if I did not move or invade this turn?', ['89.4']),
    ('exception', 'Can fighters be carried when my ship retreats if they do not fit?', ['78.7']),
    ('exception', 'What happens to ground forces being carried by a ship that disappears in a gravity rift?', ['41.2', '95.1']),
    ('exception', 'How is a tie in a vote with no votes at all resolved?', ['8.18']),
    ('exception', 'Can I use trade goods in place of resources when producing?', ['75.2', '93.3']),
    ('exception', "Does the Leadership card's second ability cost a command token for the other players?", ['52.3', '20.5']),
    ('negation', 'Can I spend command tokens from my fleet pool?', ['37.4']),
    ('negation', 'Do I have to spend a command token for the secondary ability of Leadership?', ['20.5']),
    ('negation', 'Can a damaged dreadnought sustain damage again before it is repaired?', ['87.2', '87.3']),
    ('negation', 'Can a captured fighter be handed back in a trade?', ['17.4']),
    ('negation', 'Can I put a structure on a planet that already has the maximum of that type?', ['85.5', '63.2']),
    ('negation', 'Can I move my ships into a supernova?', ['86.0']),
    ('negation', 'Can ground forces sit in space without a ship carrying them?', ['43.1']),
    ('negation', 'Can a hero be exhausted?', ['51.11']),
    ('negation', 'Can I cast votes for two different outcomes of the same agenda?', ['8.7']),
    ('negation', 'Can I research a technology that belongs to another faction?', ['90.11']),
    ('negation', 'Can I discard a secret objective I have already scored to make room?', ['61.21', '45.4']),
    ('negation', 'Can I remove the custodians token if I cannot land any troops?', ['27.2']),
    ('negation', 'Can a player who cannot vote still play a rider card?', ['FAQ-42', '8.16']),
    ('negation', 'Is it forbidden to use an exhausted planet to ignore a technology prerequisite?', ['90.15']),
    ('negation', 'Can I use Anti-Fighter Barrage in the later rounds of combat?', ['78.3']),
    ('negation', 'Are there any rules against two PDS on one planet belonging to different players?', ['63.2', '85.5']),
    ('negation', 'Is a hyperlane tile a system I can put units on?', ['44.2']),
    ('negation', 'Are the traits of a planet worth anything by themselves?', ['64.2']),
    ('role', 'Which player rolls their combat dice first in a space battle?', ['78.5']),
    ('role', 'In a three-player game how many strategy cards does each player pick?', ['84.1', '48.3']),
    ('role', 'Who resolves their ability first if several players want to act at once during the action phase?', ['1.19']),
    ('role', 'Who picks a strategy card first?', ['84.1', '80.0']),
    ('role', 'Can the defender make combat rolls before the attacker?', ['78.5']),
    ('buried', 'Do PDS units in neighboring systems fire at ships in the active system?', ['77.3']),
    ('buried', 'After a retreat do I place a command token in the system I retreated to?', ['78.7']),
    ('buried', 'Are dice rolled for units carried by a ship moving through a gravity rift?', ['41.2']),
    ('buried', 'In which order do I roll dice for ships with different combat values?', ['78.5']),
    ('buried', 'What happens to excess hits from anti-fighter barrage if I have too few fighters?', ['10.2']),
    ('buried', 'Which strategy card has initiative number 5?', ['92.0']),
    ('buried', 'How many victory points do I get for removing the custodians token?', ['27.3']),
    ('buried', 'How many planets can other players ready with the Diplomacy card?', ['32.3']),
    ('buried', 'How much does an extra technology cost the other players?', ['91.3']),
    ('buried', 'What do I gain if I score with Imperial while I control Mecatol Rex?', ['45.2']),
    ('buried', 'How many commodities can I hold at most?', ['21.1']),
    ('buried', 'How many action cards do I draw with the Politics card?', ['66.2']),
    ('buried', 'Does Plasma Scoring give each unit an extra die?', ['FAQ-4']),
    ('buried', 'If I invade two planets that both have PDS, how many extra dice does Plasma Scoring give?', ['FAQ-16']),
    ('buried', 'Can Direct Hit be played against a hit that came from my own abilities?', ['FAQ-35', 'Errata-Direct']),
    ('buried', 'Can I reveal my secret objective to another player on purpose?', ['FAQ-7']),
    ('buried', 'Can the Nekro Virus copy a printed faction unit with Valefar Assimilator?', ['FAQ-47', '90.17']),
    ('buried', 'Does the Bribery card still work if the speaker does not vote?', ['FAQ-43', 'Errata-Bribery']),
    ('negation', 'Am I forbidden from passing before I have used my strategy card?', ['3.4']),
    ('negation', "Does another player's command token stop me from activating a system?", ['5.2', '89.1']),
    ('negation', 'Can the same ability reroll one die over and over?', ['74.0']),
    ('negation', 'Am I allowed to trade promissory notes that are already in play?', ['69.5']),
    ('negation', 'Can I trade more than one promissory note at a time?', ['69.5']),
    ('negation', 'Can I keep the resources I did not spend when building units?', ['67.1']),
    ('negation', 'Can my mech be upgraded like other units?', ['55.3', '97.5']),
    ('negation', 'Can I stack two space docks on the same planet?', ['79.2', '85.2']),
    ('negation', 'Can I own more than ten victory points?', ['98.3']),
    ('negation', 'If one card says cannot and another says can, which one wins?', ['1.6']),
    ('negation', 'Am I allowed to skip the optional part of an ability?', ['1.7']),
    ('negation', 'Can I take control of a planet I already own by invading it?', ['25.2', '49.5']),
    ('negation', 'Who is not considered my opponent in a battle?', ['62.1', '62.0']),
    ('negation', 'Is it possible to land fighters instead of infantry on a planet?', ['FAQ-59', 'Errata-Matriarch']),
    ('negation', 'Can Sustain Damage stop a card that kills a unit outright?', ['87.5']),
    ('negation', 'Can I use exhausted cards to pay for things?', ['34.0', '64.8']),
    ('negation', 'May an agent or commander be taken by other players?', ['51.7']),
    ('negation', 'Can I research a unit upgrade to satisfy a prerequisite?', ['90.11', '90.7']),
    ('negation', 'Can I swap the technology of someone eliminated?', ['90.17']),
    ('negation', "Is the speaker's tie-break decision counted as a vote?", ['8.18']),
    ('role', 'Who has to announce a retreat first?', ['78.4']),
    ('role', 'Who reveals the agenda card before votes?', ['8.2', '80.2']),
    ('role', 'Who reveals the public objective each round?', ['81.2', '61.14']),
    ('role', 'Who decides who becomes speaker with the Politics card?', ['66.2', '80.6']),
    ('role', 'Is the speaker allowed to vote first?', ['80.2', '8.2']),
    ('role', 'Which player may use Space Cannon Offense?', ['77.1']),
    ('role', 'Who rolls first in ground combat?', ['42.1']),
    ('role', 'Who gets to choose which excess units are removed?', ['16.3', '37.2']),
    ('role', "Can another player use my planet's resources?", ['64.8', '75.2']),
    ('role', 'Who picks the order of resolving multiple ground combats?', ['49.4']),
]
CATEGORY = {"keyword": "Keyword trap / distractor", "exception": "Exception / corner case",
            "negation": "Negation & constraints", "role": "Role limits", "buried": "Buried specific"}

def gold_idx(gold_ids):
    return [i for i, c in enumerate(chunks) if set(re.split(r"[–#]", c["id"])) & set(gold_ids)]

rows = []
for cat, q, gold in CANDIDATES:
    g = gold_idx(gold)
    assert g, f"gold id not found for: {q}"
    qvec = embed_query(q)
    _, ids = index.search(qvec, index.ntotal)                  # full ranking: shows where gold sits even beyond rank 10
    full_rank = {int(i): r for r, i in enumerate(ids[0], start=1)}
    bi_rank = min(full_rank[i] for i in g)                     # best-placed gold chunk
    ce_rank = None
    if bi_rank <= TOP_K:                                       # the cross-encoder only sees the top-10 candidates
        ce_rank = min(c["ce_rank"] for c in rerank(q, search(qvec)) if c["idx"] in g)
    if bi_rank > TOP_K:      outcome = "gold outside top 10 (re-ranking cannot help)"
    elif bi_rank <= FINAL_K: outcome = "already in top 3" if ce_rank <= FINAL_K else "demoted out of top 3"
    else:                    outcome = "promoted into top 3" if ce_rank <= FINAL_K else "still outside top 3"
    rows.append((cat, q, gold, bi_rank, ce_rank, outcome))

disc = pd.DataFrame(rows, columns=["category", "query", "gold", "bi_rank", "ce_rank", "outcome"])
print(f"{len(disc)} candidate questions")
display(disc["outcome"].value_counts().to_frame("questions"))

failures = disc[disc["outcome"] == "promoted into top 3"].reset_index(drop=True)
print(f"\n{len(failures)} questions where the gold chunk sat at rank 4–10 and re-ranking moved it into the top 3:")
display(failures.assign(category=failures["category"].map(CATEGORY))[["category", "query", "bi_rank", "ce_rank"]])

153 candidate questions


,questions
outcome,
already in top 3,134
promoted into top 3,9
gold outside top 10 (re-ranking cannot help),7
still outside top 3,3



9 questions where the gold chunk sat at rank 4–10 and re-ranking moved it into the top 3:


,category,query,bi_rank,ce_rank
0,Keyword trap / distractor,Which planets can be explored for new cards?,5,2.0
1,Negation & constraints,Can I make a unit that has no cost?,4,1.0
2,Keyword trap / distractor,What do I do when I have more fighters than my...,7,1.0
3,Exception / corner case,What happens to structures on a planet when so...,5,1.0
4,Exception / corner case,Can fighters be carried when my ship retreats ...,9,1.0
5,Role limits,Who picks a strategy card first?,6,1.0
6,Negation & constraints,Can I research a unit upgrade to satisfy a pre...,4,1.0
7,Role limits,Who reveals the public objective each round?,4,1.0
8,Role limits,Who gets to choose which excess units are remo...,9,1.0


**Observation:** for most candidates the baseline already had the gold chunk in the top 3. The topic title is prepended to every chunk and the chunks are small and focused, so MiniLM does well when the question reuses the rulebook's words. The failures that remain come from lay wording, synonyms ("carriers can hold" vs "capacity"), many near-identical chunks competing (all the strategy-card chunks), and exceptions that live in a different topic than the general rule.

## 9. Rank-shift tables for the failure queries
For every failure query both pipelines run on the same top-10. The table lists the 10 candidates in baseline order with bi-encoder cosine and cross-encoder score; `shift = baseline rank − re-ranked rank` (positive means promoted). The gold chunk is highlighted. These runs also call the LLM for both pipelines (Section 10 compares the answers).

In [10]:
from IPython.display import display, Markdown

results, summary_lines = {}, []
for n, row in failures.iterrows():
    res = run_both(row["query"])
    results[row["query"]] = res
    tbl = rank_table(res)
    gold_names = {chunks[i]["name"] for i in gold_idx(row["gold"])}
    best = tbl[tbl["chunk"].isin(gold_names)].sort_values("ce_rank").iloc[0]
    line = f'Chunk "{best["chunk"]}": rank #{int(best["bi_rank"])} → #{int(best["ce_rank"])}'
    summary_lines.append((row["query"], line))
    display(Markdown(f'**Q{n+1} [{CATEGORY[row["category"]]}]** {row["query"]}  \n{line}'))
    display(tbl.style.apply(lambda r: ["background-color: #fff3b0" if r["chunk"] in gold_names else "" for _ in r], axis=1)
               .format({"bi_score": "{:.3f}", "ce_score": "{:.2f}"}).hide(axis="index"))

**Q1 [Keyword trap / distractor]** Which planets can be explored for new cards?  
Chunk "35.2 Exploration": rank #5 → #2

chunk,bi_rank,bi_score,ce_rank,ce_score,shift
35.0–35.1 Exploration,1,0.619,1,6.27,0
35.8 Exploration,2,0.573,3,4.10,-1
Setup-Step5 Game Setup,3,0.566,8,-2.01,-5
64.6 Planets,4,0.545,5,0.50,-1
35.2 Exploration,5,0.504,2,4.40,3
35.3–35.4 Exploration,6,0.466,4,1.26,2
12.3 Attach,7,0.463,9,-2.71,-2
47.1–47.2 Influence,8,0.452,7,-1.57,1
64.7 Planets,9,0.452,6,-0.50,3
Setup-Step9 Game Setup,10,0.442,10,-5.35,0


**Q2 [Negation & constraints]** Can I make a unit that has no cost?  
Chunk "26.2–26.3 Cost (Attribute)": rank #4 → #1

chunk,bi_rank,bi_score,ce_rank,ce_score,shift
FAQ-17 Production,1,0.567,5,-2.41,-4
67.1 Producing Units,2,0.561,2,-1.08,0
26.1 Cost (Attribute),3,0.545,3,-1.17,0
26.2–26.3 Cost (Attribute),4,0.538,1,4.36,3
67.2 Producing Units,5,0.522,4,-1.30,1
26.0 Cost (Attribute),6,0.468,6,-4.58,0
FAQ-18 Production,7,0.437,7,-5.13,0
68.1 Production (Unit Ability),8,0.397,8,-5.29,0
67.4 Producing Units,9,0.386,10,-9.24,-1
67.5 Producing Units,10,0.367,9,-6.05,1


**Q3 [Keyword trap / distractor]** What do I do when I have more fighters than my carriers can hold after the battle?  
Chunk "16.3 Capacity (Attribute)": rank #7 → #1

...

chunk,bi_rank,bi_score,ce_rank,ce_score,shift
97.4 Unit Upgrades,1,0.486,3,-3.44,-2
97.5 Unit Upgrades,2,0.453,8,-6.15,-6
90.6 Technology,3,0.416,5,-4.86,-2
90.11–90.12 Technology,4,0.415,1,6.68,3
97.2–97.3 Unit Upgrades,5,0.405,6,-4.86,-1
97.0–97.1 Unit Upgrades,6,0.402,2,-2.32,4
FAQ-50 The Nekro Virus,7,0.375,4,-4.68,3
90.21 Technology,8,0.371,7,-5.13,1
FAQ-51 The Nekro Virus,9,0.362,9,-6.59,0
90.0 Technology,10,0.344,10,-7.75,0


**Q8 [Role limits]** Who reveals the public objective each round?  
Chunk "81.2 Status Phase": rank #4 → #1

chunk,bi_rank,bi_score,ce_rank,ce_score,shift
61.11–61.12 Objective Cards,1,0.654,4,4.06,-3
61.15 Objective Cards,2,0.595,3,4.55,-1
61.16–61.17 Objective Cards,3,0.538,5,0.28,-2
81.2 Status Phase,4,0.517,1,4.73,3
61.18 Objective Cards,5,0.497,6,-0.00,-1
61.14 Objective Cards,6,0.495,2,4.73,4
61.0–61.1 Objective Cards,7,0.495,7,-0.30,0
61.22 Objective Cards,8,0.494,10,-8.30,-2
61.6–61.7 Objective Cards,9,0.481,8,-1.24,1
Setup-Step12 Game Setup,10,0.477,9,-4.11,1


**Q9 [Role limits]** Who gets to choose which excess units are removed?  
Chunk "16.3 Capacity (Attribute)": rank #9 → #1

chunk,bi_rank,bi_score,ce_rank,ce_score,shift
96.3 Units,1,0.593,9,-8.71,-8
67.5 Producing Units,2,0.553,2,-0.81,0
23.3–23.4 Component Limitations,3,0.522,3,-0.82,0
67.2 Producing Units,4,0.519,8,-8.07,-4
68.1 Production (Unit Ability),5,0.509,10,-9.45,-5
61.8–61.9 Objective Cards,6,0.499,5,-3.25,1
33.11 Elimination,7,0.491,4,-2.04,3
67.1 Producing Units,8,0.487,6,-7.07,2
16.3 Capacity (Attribute),9,0.482,1,6.34,8
97.4 Unit Upgrades,10,0.463,7,-8.07,3


## 10. Answer 1 vs Answer 2
Same prompt, same model, temperature 0. Only the 3 chunks differ. After each pair I recorded a **manual verdict** on how the baseline answer went wrong and whether the re-ranked answer is right. The verdicts were written after reading the output of the recorded run.

In [11]:
VERDICTS = {
    'Which planets can be explored for new cards?':
        'Answer 1 omits the exception that planets without traits (Mecatol Rex, home planets) cannot be explored. Answer 2 had the exception chunk (35.2) but the model INVERTED it ("do not have traits ... can be explored"), so it is also wrong. Retrieval fixed, generation still failed.',
    'Can I make a unit that has no cost?':
        'Answer 1 omits the rule "if a unit does not have a cost, it cannot be produced" (26.2) and gives no answer. Answer 2 is correct.',
    'What do I do when I have more fighters than my carriers can hold after the battle?':
        'Answer 1 omits the capacity rule (16.3) and gives no answer. Answer 2 is correct: remove the excess units.',
    'What happens to structures on a planet when someone else takes control of it?':
        "Answer 1 omits the clause in 49.5(a) and says the excerpts do not cover it. Answer 2 is correct: other players' structures are destroyed.",
    'Can fighters be carried when my ship retreats if they do not fit?':
        'Answer 1 omits the clause in 78.7(b) and gives no answer, and it cites [37.3 Fleet Pool], a chunk that was not in its context: a hallucinated citation, copied from the example id in the prompt. Answer 2 reaches the right conclusion (No) with wrong reasoning; the rule says such fighters are removed. Partially correct.',
    'Who picks a strategy card first?':
        'Answer 1 is distracted by the many strategy-card chunks and gives no answer. Answer 2 is correct: the speaker, then clockwise.',
    'Can I research a unit upgrade to satisfy a prerequisite?':
        'Answer 1 gives no answer. Answer 2 had the right chunk (90.11-90.12: unit upgrades do not satisfy prerequisites) but MISSTATES the rule ("Yes"). Retrieval fixed, generation failed.',
    'Who reveals the public objective each round?':
        'Answer 1 omits 81.2 and gives no answer. Answer 2 is correct: the speaker, in the status phase.',
    'Who gets to choose which excess units are removed?':
        'Answer 1 wrongly says it is not specified (the answer is in 16.3(a)). Answer 2 is correct: the player who owns the units chooses.',
}
for n, (q, res) in enumerate(results.items(), start=1):
    print(f"Q{n}. {q}")
    print("  baseline top-3 :", [chunks[i]["name"] for i in res["top3_base"]])
    print("  re-ranked top-3:", [chunks[i]["name"] for i in res["top3_rr"]])
    print("  Answer 1 (baseline):", res["answer1"].replace("\n", " "))
    print("  Answer 2 (re-rank) :", res["answer2"].replace("\n", " "))
    print("  verdict:", VERDICTS.get(q, "(not assessed)"), "\n")

Q1. Which planets can be explored for new cards?
  baseline top-3 : ['35.0–35.1 Exploration', '35.8 Exploration', 'Setup-Step5 Game Setup']
  re-ranked top-3: ['35.0–35.1 Exploration', '35.2 Exploration', '35.8 Exploration']
  Answer 1 (baseline): Planets that are not already controlled by another player can be explored for new cards [35.0–35.1 Exploration].
  Answer 2 (re-rank) : Planets that are not already controlled by another player and do not have traits (such as Mecatol Rex or planets in home systems) can be explored for new cards. [35.0–35.1 Exploration], [35.2 Exploration]
  verdict: Answer 1 omits the exception that planets without traits (Mecatol Rex, home planets) cannot be explored. Answer 2 had the exception chunk (35.2) but the model INVERTED it ("do not have traits ... can be explored"), so it is also wrong. Retrieval fixed, generation still failed. 

Q2. Can I make a unit that has no cost?
  baseline top-3 : ['FAQ-17 Production', '67.1 Producing Units', '26.1 Cost (Att

## 11. Latency profiling
Timings use `time.perf_counter()` with the GPU synchronised. A **warm-up** is run first, because the first calls pay one-off costs (CUDA kernels, loading the LLM into GPU memory) that would distort the numbers.

* **Table A – retrieval only**, over all 153 candidate questions: the baseline pipeline is *query embedding + FAISS search*; the re-ranking pipeline adds the *cross-encoder on 10 pairs*.
* **Table B – end to end**, over the 9 failure queries × 3 repeats, including LLM generation (the LLM call is the same size in both pipelines because both receive 3 chunks). Retrieval is slower here than in Table A because each retrieval runs right after an LLM call, while Ollama still holds the GPU.

In [12]:
for _ in range(3):
    run_both("warm-up question about the fleet pool", with_llm=False)
generate("warm-up", [0])                       # make sure the LLM is loaded in GPU memory

ms = lambda v: f"{np.mean(v)*1000:.2f} ± {np.std(v)*1000:.2f}"

# ---- Table A: retrieval only ----
tA = [run_both(q, with_llm=False)["t"] for _, q, _ in CANDIDATES]
emb_t = [t["embed"] for t in tA]; srch_t = [t["search"] for t in tA]; ce_t = [t["cross_encoder"] for t in tA]
base_ret = [a + b for a, b in zip(emb_t, srch_t)]
rr_ret   = [a + b + c for a, b, c in zip(emb_t, srch_t, ce_t)]
latA = pd.DataFrame({
    "Stage": ["Query embedding (bi-encoder)", "FAISS top-10 search", "Cross-encoder, 10 pairs", "Retrieval total"],
    "Baseline (ms)":     [ms(emb_t), ms(srch_t), "–", ms(base_ret)],
    "Re-ranking (ms)":   [ms(emb_t), ms(srch_t), ms(ce_t), ms(rr_ret)]})
print(f"Table A: retrieval latency, mean ± std over {len(tA)} queries")
display(latA)
print(f"Re-ranking multiplies retrieval time by {np.mean(rr_ret)/np.mean(base_ret):.1f}x (adds {np.mean(ce_t)*1000:.1f} ms).")

# ---- Table B: end to end ----
tB = [run_both(q)["t"] for q in failures["query"] for _ in range(3)]
r1 = [t["embed"] + t["search"] for t in tB];                       l1 = [t["llm1"] for t in tB]
r2 = [t["embed"] + t["search"] + t["cross_encoder"] for t in tB];  l2 = [t["llm2"] for t in tB]
latB = pd.DataFrame({
    "Stage": ["Retrieval", "LLM generation", "End-to-end"],
    "Pipeline 1, baseline (ms)":    [ms(r1), ms(l1), ms([a + b for a, b in zip(r1, l1)])],
    "Pipeline 2, re-ranking (ms)":  [ms(r2), ms(l2), ms([a + b for a, b in zip(r2, l2)])]})
print(f"\nTable B: end-to-end latency, mean ± std over {len(tB)} runs")
display(latB)

Table A: retrieval latency, mean ± std over 153 queries


,Stage,Baseline (ms),Re-ranking (ms)
0,Query embedding (bi-encoder),2.35 ± 0.17,2.35 ± 0.17
1,FAISS top-10 search,0.06 ± 0.01,0.06 ± 0.01
2,"Cross-encoder, 10 pairs",–,9.88 ± 3.64
3,Retrieval total,2.41 ± 0.17,12.28 ± 3.65


Re-ranking multiplies retrieval time by 5.1x (adds 9.9 ms).



Table B: end-to-end latency, mean ± std over 27 runs


,Stage,"Pipeline 1, baseline (ms)","Pipeline 2, re-ranking (ms)"
0,Retrieval,3.60 ± 0.29,19.10 ± 5.12
1,LLM generation,970.59 ± 342.70,890.08 ± 258.92
2,End-to-end,974.20 ± 342.87,909.17 ± 260.09


## 12. Why not run the cross-encoder over the whole collection?
A cross-encoder needs a full transformer pass for **every** `(query, chunk)` pair, and nothing can be precomputed because the query is part of the input. The cost per question therefore grows linearly with the collection size. The bi-encoder embeds the chunks once, offline; at query time it only encodes the query and does a vector search (which an ANN index makes sublinear). Below, I measure the cross-encoder on all 511 chunks for one query and extrapolate.

In [13]:
q = CANDIDATES[0][1]
pairs = [(q, c["text"]) for c in chunks]
ce.predict(pairs[:64], batch_size=64, show_progress_bar=False)                 # warm-up
_, t_all = timed(lambda: ce.predict(pairs, batch_size=64, show_progress_bar=False))
per_pair = t_all / len(pairs)
scale = pd.DataFrame({
    "Collection size (chunks)": [TOP_K, len(chunks), 100_000, 1_000_000],
    "Cross-encoder time for one query": [f"{per_pair*n:.3f} s" for n in (TOP_K, len(chunks), 100_000, 1_000_000)],
    "Source": ["measured per-pair cost × 10 (this is our re-ranking step)", "measured (all chunks, one query)",
               "extrapolated linearly", "extrapolated linearly"]})
display(scale)
print(f"Cross-encoder on all {len(chunks)} chunks: {t_all:.2f} s per query vs {np.mean(base_ret)*1000:.1f} ms for baseline retrieval.")
print("(extrapolation ignores batching/memory effects on a real GPU and a bigger corpus; it only shows the order of magnitude)")

,Collection size (chunks),Cross-encoder time for one query,Source
0,10,0.010 s,measured per-pair cost × 10 (this is our re-ra...
1,511,0.507 s,"measured (all chunks, one query)"
2,100000,99.197 s,extrapolated linearly
3,1000000,991.975 s,extrapolated linearly


Cross-encoder on all 511 chunks: 0.51 s per query vs 2.4 ms for baseline retrieval.
(extrapolation ignores batching/memory effects on a real GPU and a bigger corpus; it only shows the order of magnitude)


## 13. Export results
CSVs for the repo, and LaTeX tables for the report.

In [14]:
disc.assign(gold=disc["gold"].map(", ".join)).to_csv("results/candidate_ranks.csv", index=False)
long = []
for n, (q, res) in enumerate(results.items(), start=1):
    long.append(rank_table(res).assign(query_no=n, query=q))
pd.concat(long).to_csv("results/failure_rank_shifts.csv", index=False)
pd.DataFrame([{"query": q, "category": CATEGORY[failures.loc[n, 'category']], "baseline_top3": " | ".join(chunks[i]["name"] for i in r["top3_base"]),
               "reranked_top3": " | ".join(chunks[i]["name"] for i in r["top3_rr"]), "answer1": r["answer1"], "answer2": r["answer2"],
               "verdict": VERDICTS.get(q, "")} for n, (q, r) in enumerate(results.items())]).to_csv("results/answers.csv", index=False)
latA.to_csv("results/latency_retrieval.csv", index=False); latB.to_csv("results/latency_end_to_end.csv", index=False)
scale.to_csv("results/scaling.csv", index=False)

# LaTeX tables (used by the reference report)
for n, (q, res) in enumerate(results.items(), start=1):
    t = rank_table(res).rename(columns={"chunk": "Chunk", "bi_rank": "Bi rank", "bi_score": "Cosine", "ce_rank": "CE rank", "ce_score": "CE score", "shift": "Shift"})
    t.to_latex(f"report/tables/rank_shift_q{n}.tex", index=False, escape=True, float_format="%.2f")
latA.to_latex("report/tables/latency_retrieval.tex", index=False, escape=True)
latB.to_latex("report/tables/latency_end_to_end.tex", index=False, escape=True)
scale.drop(columns="Source").to_latex("report/tables/scaling.tex", index=False, escape=True)
print("saved:", sorted(p.name for p in Path("results").iterdir()), "+ LaTeX tables in report/tables/")

saved: ['answers.csv', 'candidate_ranks.csv', 'failure_rank_shifts.csv', 'latency_end_to_end.csv', 'latency_retrieval.csv', 'scaling.csv'] + LaTeX tables in report/tables/
